# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [1]:
import pandas as pd
import os

## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, date_start, date_end, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, and either `date` (single-day talks) or `date_start` (multi-day talks). All else can be blank. `type` defaults to "Talk"
- Dates must be formatted as YYYY-MM-DD.
- Use `date` for a talk on a single day. Use `date_start` and `date_end` for a talk spanning several days; the site then renders the date as a range.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the talk.
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talk/YYYY-MM-DD-[url_slug]`, where YYYY-MM-DD is `date_start` if it is set and `date` otherwise
    - The combination of `url_slug` and that date must be unique, as it is the basis for both the filename and the permalink

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [2]:
!cat talks.tsv

title	type	url_slug	venue	date	date_start	date_end	location	talk_url	description
Fast potential flow computations for low-order aerodynamic modeling	Conference presentation	aps-72	72nd Annual Meeting of the APS Division of Fluid Dynamics		2019-11-23	2019-11-26	"Seattle, WA"		
Fast Potential Flow Computations for Low-order Aerodynamic Modeling	Conference presentation	socal-14	14th Southern California Flow Physics Symposium	2020-04-10			Virtual		
Fast Potential Flow Computations for Low-order Aerodynamic Modeling	Conference presentation	aps-73	73rd Annual Meeting of the APS Division of Fluid Dynamics		2020-11-22	2020-11-24	Virtual		
Aggregated Lifting-line Vortex Modeling for Unsteady Aerodynamics	Conference presentation	aps-74	74th Annual Meeting of the APS Division of Fluid Dynamics		2020-11-26	2020-11-29	"Phoenix, AZ"		
Towards a computationally-augmented wind tunnel for unsteady aerodynamics	Conference presentation	socal-15	15th Southern California Flow Physics Symposium	2022-04-23		

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
talks = pd.read_csv("talks.tsv", sep="\t", header=0)
talks

,title,type,url_slug,venue,date,date_start,date_end,location,talk_url,description
0,Fast potential flow computations for low-order...,Conference presentation,aps-72,72nd Annual Meeting of the APS Division of Flu...,NaN,2019-11-23,2019-11-26,"Seattle, WA",NaN,NaN
1,Fast Potential Flow Computations for Low-order...,Conference presentation,socal-14,14th Southern California Flow Physics Symposium,2020-04-10,NaN,NaN,Virtual,NaN,NaN
2,Fast Potential Flow Computations for Low-order...,Conference presentation,aps-73,73rd Annual Meeting of the APS Division of Flu...,NaN,2020-11-22,2020-11-24,Virtual,NaN,NaN
3,Aggregated Lifting-line Vortex Modeling for Un...,Conference presentation,aps-74,74th Annual Meeting of the APS Division of Flu...,NaN,2020-11-26,2020-11-29,"Phoenix, AZ",NaN,NaN
4,Towards a computationally-augmented wind tunne...,Conference presentation,socal-15,15th Southern California Flow Physics Symposium,2022-04-23,NaN,NaN,"Los Angeles, CA",NaN,NaN
5,Discretization error analysis of convective sc...,Conference presentation,nrel,Rocky Mountain Fluid Mechanics Research Symposium,2022-08-09,NaN,NaN,"Boulder, CO",NaN,NaN
6,A computationally-augmented wind tunnel with i...,Conference presentation,aps-75,75th Annual Meeting of the APS Division of Flu...,NaN,2022-11-20,2022-11-22,"Indianapolis, IN",NaN,NaN
7,Deep reinforcement learning of airfoil pitch c...,Conference presentation,socal-16,16th Southern California Flow Physics Symposium,2023-04-22,NaN,NaN,"San Diego, CA",NaN,NaN
8,Deep reinforcement learning of airfoil pitch c...,Conference presentation,discovor-2,2nd Direct in-Person Colloquium on Vortex Domi...,NaN,2023-05-16,2023-05-19,"Breckenridge, CO",NaN,NaN
9,Deep reinforcement learning of airfoil pitch c...,Conference presentation,aps-76,76th Annual Meeting of the APS Division of Flu...,NaN,2023-11-19,2023-11-21,"Washington, D.C.",NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [ ]:
def field(item, name):
    """Value of a TSV column as a stripped string, or None when it is empty."""
    value = item.get(name)
    if pd.isna(value):
        return None
    return str(value).strip() or None

for row, item in talks.iterrows():

    date_start = field(item, "date_start")
    date_end = field(item, "date_end")
    date_single = field(item, "date")

    # Multi-day talks give a start and end date, single-day talks just a date.
    # The filename and the permalink are both built from this one slug, so they
    # can never drift apart.
    slug = str(date_start or date_single) + "-" + item.url_slug

    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    md += 'type: "' + (field(item, "type") or "Talk") + '"\n'
    md += "permalink: /talk/" + slug + "\n"

    if field(item, "venue"):
        md += 'venue: "' + field(item, "venue") + '"\n'

    if date_start:
        md += "date_start: " + date_start + "\n"
        if date_end:
            md += "date_end: " + date_end + "\n"
    elif date_single:
        md += "date: " + date_single + "\n"

    if field(item, "location"):
        md += 'location: "' + field(item, "location") + '"\n'

    md += "---\n"


    if field(item, "talk_url"):
        md += "\n[More information here](" + field(item, "talk_url") + ")\n"


    if field(item, "description"):
        md += "\n" + html_escape(field(item, "description")) + "\n"


    md_filename = os.path.basename(slug + ".md")
    #print(md)

    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [6]:
!ls ../_talks

2012-03-01-talk-1.md	  2014-02-01-talk-2.md
2013-03-01-tutorial-1.md  2014-03-01-talk-3.md


In [7]:
!cat ../_talks/2013-03-01-tutorial-1.md

---
title: "Tutorial 1 on Relevant Topic in Your Field"
collection: talks
type: "Tutorial"
permalink: /talks/2013-03-01-tutorial-1
venue: "UC-Berkeley Institute for Testing Science"
date: 2013-03-01
location: "Berkeley CA, USA"
---

[More information here](http://exampleurl.com)

This is a description of your tutorial, note the different field in type. This is a markdown files that can be all markdown-ified like any other post. Yay markdown!
